# Canteen IQ: AI Analysis Accuracy vs. Proprietor Baseline

This notebook evaluates the performance of **Canteen IQ** (Multi-Agent System) against a **Proprietor Baseline** (Intuition-based 'Usual Order') over a 150-day semester at PES University.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load the generated data
df = pd.read_csv('semester_results.csv')
df['Date'] = pd.to_datetime(df['Date'])

# 1. Calculate Error Metrics
def calculate_metrics(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return mae, mape

iq_mae, iq_mape = calculate_metrics(df['Actual_Demand'], df['CanteenIQ_Prediction'])
base_mae, base_mape = calculate_metrics(df['Actual_Demand'], df['Baseline_Order'])

print(f"--- Accuracy Analysis ---")
print(f"Canteen IQ MAE: {iq_mae:.2f} units | MAPE: {iq_mape:.2f}%")
print(f"Proprietor Baseline MAE: {base_mae:.2f} units | MAPE: {base_mape:.2f}%")
print(f"Accuracy Improvement: {((base_mae - iq_mae) / base_mae * 100):.2f}% improvement in error reduction.")

### Comparison Graph: Actual vs. AI vs. Baseline
The graph below highlights how **Canteen IQ** tracks demand spikes (Exams/Rain) that the **Baseline** misses.

In [ ]:
plt.figure(figsize=(16, 7))
plt.plot(df['Date'], df['Actual_Demand'], label='Actual Demand', color='#374175', alpha=0.3, linewidth=1)
plt.plot(df['Date'], df['CanteenIQ_Prediction'], label='Canteen IQ (AI)', color='#ee8326', linewidth=2.5)
plt.plot(df['Date'], df['Baseline_Order'], label='Proprietor Baseline (Intuition)', color='#94a3b8', linestyle='--', linewidth=1.5)

# Highlight an exam week
plt.axvspan(pd.Timestamp('2026-02-15'), pd.Timestamp('2026-02-22'), color='orange', alpha=0.1, label='ISA Exam Week')

plt.title('Performance Comparison: Canteen IQ AI vs. Proprietor Intuition', fontsize=16, pad=20)
plt.xlabel('Semester Timeline (Jan - May 2026)', fontsize=12)
plt.ylabel('Order Quantity (Units)', fontsize=12)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

### Waste Reduction Analysis
Excess inventory occurs when the order is greater than the actual demand. Canteen IQ significantly reduces this 'Over-ordering' during holidays and weekends.

In [ ]:
df['IQ_Waste'] = (df['CanteenIQ_Prediction'] - df['Actual_Demand']).clip(lower=0)
df['Base_Waste'] = (df['Baseline_Order'] - df['Actual_Demand']).clip(lower=0)

waste_improvement = (df['Base_Waste'].sum() - df['IQ_Waste'].sum()) / df['Base_Waste'].sum() * 100

plt.figure(figsize=(10, 6))
sns.barplot(x=['Proprietor Baseline', 'Canteen IQ'], y=[df['Base_Waste'].sum(), df['IQ_Waste'].sum()], palette=['#94a3b8', '#ee8326'])
plt.title(f'Cumulative Perishable Waste Reduction: {waste_improvement:.1f}% Improvement')
plt.ylabel('Total Units Wasted')
plt.show()